<a href="https://colab.research.google.com/github/kale-abhijeet/extract_Data/blob/main/pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, countDistinct, month, year, max, to_date, datediff, current_date

In [ ]:
spark = SparkSession.builder \
        .appName("Retail POC") \
        .master("local[*]") \
        .getOrCreate()

In [ ]:
spark

In [ ]:
#load Dataframe
df_customers = spark.read.option("header", True).option("inferSchema", True).csv("/content/customers.csv")
df_transactions = spark.read.option("header", True).option("inferSchema", True).csv("/content/transactions.csv")
df_products = spark.read.option("header", True).option("inferSchema", True).csv("/content/products.csv")

In [ ]:
df_customers.show()

+-----------+------------+--------------------+-----------+
|customer_id|        name|               email|signup_date|
+-----------+------------+--------------------+-----------+
|        101|Customer_101|customer101@examp...| 2022-10-17|
|        102|Customer_102|customer102@examp...| 2021-04-25|
|        103|Customer_103|customer103@examp...| 2021-01-26|
|        104|Customer_104|customer104@examp...| 2023-01-30|
|        105|Customer_105|customer105@examp...| 2021-10-09|
|        106|Customer_106|customer106@examp...| 2021-09-08|
|        107|Customer_107|customer107@examp...| 2021-08-17|
|        108|Customer_108|customer108@examp...| 2021-05-23|
|        109|Customer_109|customer109@examp...| 2023-01-25|
|        110|Customer_110|customer110@examp...| 2021-04-15|
|        111|Customer_111|customer111@examp...| 2022-11-24|
|        112|Customer_112|customer112@examp...| 2023-01-29|
|        113|Customer_113|customer113@examp...| 2023-07-03|
|        114|Customer_114|customer114@ex

In [ ]:
df_products.show()

+----------+------------+--------------+
|product_id|product_name|      category|
+----------+------------+--------------+
|      2001|      Laptop|   Electronics|
|      2002|  Headphones|   Electronics|
|      2003|Coffee Maker|Home Appliance|
|      2004|  Desk Chair|     Furniture|
|      2005|  Smartphone|   Electronics|
|      2006|     Blender|Home Appliance|
|      2007|     Monitor|   Electronics|
|      2008|   Air Fryer|Home Appliance|
+----------+------------+--------------+



In [ ]:
df_transactions.show()

+--------------+-----------+----------+-------+----------+
|transaction_id|customer_id|product_id| amount| timestamp|
+--------------+-----------+----------+-------+----------+
|          1001|        122|      2002| 184.48|2023-04-10|
|          1002|        123|      2006|  925.4|2023-02-14|
|          1003|        147|      2008| 827.53|2024-01-23|
|          1004|        106|      2005|1252.64|2024-01-06|
|          1005|        137|      2004|1071.63|2023-02-16|
|          1006|        143|      2004|1170.95|2023-03-23|
|          1007|        115|      2002| 601.18|2024-04-09|
|          1008|        141|      2006| 285.85|2023-12-30|
|          1009|        114|      2005|1067.64|2023-03-15|
|          1010|        139|      2003|  824.5|2023-09-08|
|          1011|        111|      2008| 600.21|2024-07-24|
|          1012|        115|      2006|1272.14|2023-02-27|
|          1013|        115|      2001|1217.32|2024-02-15|
|          1014|        118|      2002| 355.93|2024-08-0

In [ ]:
# Convert timestamp to proper date format
df_transactions = df_transactions.withColumn("date", to_date(col("timestamp")))

In [ ]:
# Join transactions with customer and product data
df_full = df_transactions.join(df_customers, "customer_id", "left") \
                         .join(df_products, "product_id", "left")

# ========================== KPIs ==========================

In [ ]:
# 1. Total Revenue per Customer
revenue_by_customer = df_full.groupBy("customer_id").agg(sum("amount").alias("total_revenue")).orderBy(col("total_revenue").desc())

In [ ]:
print("Total Revenue per Customer:")
revenue_by_customer.show()


Total Revenue per Customer:
+-----------+------------------+
|customer_id|     total_revenue|
+-----------+------------------+
|        115|            4453.5|
|        114|3250.2799999999997|
|        106|           2363.91|
|        143|            1856.2|
|        108|           1640.21|
|        123|            1571.5|
|        116|           1532.43|
|        133|           1494.07|
|        137|           1473.98|
|        101|           1459.24|
|        134|           1458.06|
|        139|1432.4099999999999|
|        149|           1400.64|
|        102|           1397.19|
|        130|            1318.0|
|        129|           1194.54|
|        144|           1176.37|
|        122|           1087.53|
|        135|            918.47|
|        147|            827.53|
+-----------+------------------+
only showing top 20 rows



In [ ]:
from ast import alias
# 2. Monthly Revenue Trends
monthly_revenue = df_full.groupBy(year("date"), month("date")) \
                    .agg(sum("amount").alias("monthly_revenue")) \
                    .orderBy(year("date"), month("date"))

In [ ]:
monthly_revenue.show()

+----------+-----------+---------------+
|year(date)|month(date)|monthly_revenue|
+----------+-----------+---------------+
|      2023|          1|        1458.06|
|      2023|          2|        4097.09|
|      2023|          3|        5137.75|
|      2023|          4|         184.48|
|      2023|          5|         832.05|
|      2023|          6|         271.62|
|      2023|          7|        1459.24|
|      2023|          8|         903.05|
|      2023|          9|          824.5|
|      2023|         10|         661.73|
|      2023|         11|        2709.44|
|      2023|         12|        2951.07|
|      2024|          1|        2080.17|
|      2024|          2|         2301.1|
|      2024|          3|        2714.07|
|      2024|          4|        2291.85|
|      2024|          5|        1966.74|
|      2024|          6|         685.25|
|      2024|          7|        2089.85|
|      2024|          8|        1854.05|
+----------+-----------+---------------+



In [ ]:
# 3. Most Popular Products
populer_product =  df_full.groupBy("product_name", "product_id", "category") \
                  .agg(countDistinct("transaction_id").alias("unit_solds")) \
                  .orderBy(col("unit_solds").desc())

In [ ]:
populer_product.show()

+------------+----------+--------------+----------+
|product_name|product_id|      category|unit_solds|
+------------+----------+--------------+----------+
|  Headphones|      2002|   Electronics|        12|
|  Desk Chair|      2004|     Furniture|         9|
|  Smartphone|      2005|   Electronics|         7|
|Coffee Maker|      2003|Home Appliance|         6|
|   Air Fryer|      2008|Home Appliance|         5|
|     Blender|      2006|Home Appliance|         5|
|      Laptop|      2001|   Electronics|         4|
|     Monitor|      2007|   Electronics|         2|
+------------+----------+--------------+----------+



In [ ]:
# 4. Churned Customers (no transaction in last 90 days)
latest_transaction = df_full.groupBy("customer_id") \
    .agg(max("date").alias("last_purchase"))

In [ ]:
churned_customers = latest_transaction.filter(datediff(current_date(), col("last_purchase")) > 90)

In [ ]:
print("Churned Customers (no transaction in last 90 days):")
churned_customers.show()

Churned Customers (no transaction in last 90 days):
+-----------+-------------+
|customer_id|last_purchase|
+-----------+-------------+
|        137|   2024-04-29|
|        133|   2023-11-02|
|        108|   2024-04-09|
|        115|   2024-04-09|
|        126|   2023-05-23|
|        101|   2023-07-02|
|        122|   2023-08-24|
|        111|   2024-07-24|
|        146|   2024-05-26|
|        139|   2024-04-24|
|        127|   2024-03-17|
|        107|   2024-07-03|
|        114|   2024-08-07|
|        130|   2023-03-19|
|        136|   2024-03-09|
|        129|   2024-07-07|
|        102|   2023-11-11|
|        143|   2024-06-13|
|        141|   2023-12-30|
|        105|   2024-05-12|
+-----------+-------------+
only showing top 20 rows



In [ ]:
# Show outputs
print("🔹 Revenue by Customer:")
revenue_by_customer.show()

print("🔹 Monthly Revenue Trends:")
monthly_revenue.show()

print("🔹 Product Popularity:")
populer_product.show()

print("🔹 Churned Customers:")
churned_customers.show()

🔹 Revenue by Customer:
+-----------+------------------+
|customer_id|     total_revenue|
+-----------+------------------+
|        115|            4453.5|
|        114|3250.2799999999997|
|        106|           2363.91|
|        143|            1856.2|
|        108|           1640.21|
|        123|            1571.5|
|        116|           1532.43|
|        133|           1494.07|
|        137|           1473.98|
|        101|           1459.24|
|        134|           1458.06|
|        139|1432.4099999999999|
|        149|           1400.64|
|        102|           1397.19|
|        130|            1318.0|
|        129|           1194.54|
|        144|           1176.37|
|        122|           1087.53|
|        135|            918.47|
|        147|            827.53|
+-----------+------------------+
only showing top 20 rows

🔹 Monthly Revenue Trends:
+----------+-----------+---------------+
|year(date)|month(date)|monthly_revenue|
+----------+-----------+---------------+
|      202

In [ ]:
# Save to output directory
revenue_by_customer.coalesce(1).write.mode("overwrite").option("header", True).csv("output/revenue_by_customer")
monthly_revenue.coalesce(1).write.mode("overwrite").option("header", True).csv("output/monthly_revenue")
populer_product.coalesce(1).write.mode("overwrite").option("header", True).csv("output/popular_products")
churned_customers.coalesce(1).write.mode("overwrite").option("header", True).csv("output/churned_customers")
